# Day 9 — ILT 3: CDF-Based Incremental Loading (Bronze to Silver via MERGE)

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | ILT 2 — Watermark-Based Incremental Loading |
| **Duration** | 90 minutes |
| **Format** | Instructor-led — this enables real CDF on `<your-catalog>.bronze`/`<your-catalog>.silver` tables (safe, idempotent, no compute cost beyond a metadata change), then walks through GlobalMart's real Bronze → Silver MERGE code as a read-only worked example. Code cells below use the literal `harsh_kumar01_npmentorskool_onmicrosoft_com` catalog — GlobalMart's real Bronze/Silver run; your own catalog will be named differently. (Gold is a *separate* catalog, `gbmart` — not used anywhere in this session.) |

### Learning Objectives
- Explain what Change Data Feed actually records, and why it closes the delete blind spot from ILT 2
- Enable CDF on Bronze and Silver tables the same way GlobalMart's real pipeline does
- Read a CDF changelog two ways — `readChangeFeed` (PySpark) and `table_changes()` (SQL) — and understand `_change_type`, `_commit_version`, `_commit_timestamp`
- Walk through GlobalMart's **real** `dim_product` SCD2 MERGE — the actual code, including a real rough edge worth discussing — and see the same pattern applied a second time to `dim_customer`
- Know exactly why this ILT reads the MERGE code instead of running it, and where the hands-on, corrected version gets built (Day 10)

---

## Why CDF, Not Just "Read The Whole Table Again"

```
Without CDF:  Silver reads all 126,000+ orders every run  →  slow, expensive, gets worse over time
With CDF:     Silver reads the ~800 rows that changed since last run  →  fast, cheap, constant cost
```

CDF is a Delta Lake table property. Once enabled, **every** `INSERT` / `UPDATE` / `DELETE` against that table is automatically recorded as a row-level event — no separate logging code needed, Delta does it as part of the transaction itself.

## What CDF Actually Records

| Column | Description |
|---|---|
| `_change_type` | `insert`, `update_preimage`, `update_postimage`, or `delete` |
| `_commit_version` | The Delta table version the change happened in |
| `_commit_timestamp` | When the change was committed |

Note the two `update_*` types — an `UPDATE` produces **two** CDF rows: the row's old values (`update_preimage`) and its new values (`update_postimage`). When you only care about "what does this row look like now," filter out `update_preimage` (you'll see this exact filter in HOL 2).

In [ ]:
# Enable CDF on every Bronze table -- this matches the REAL notebook GlobalMart's
# pipeline runs (01_bronze_enable_cdf.py), verbatim in shape. Notice it does NOT
# pre-filter orders/order_items out of the list before trying. It attempts ALTER
# TABLE against all 8 tables and lets try/except record what actually happens.
# The real notebook's own markdown *predicts* orders/order_items will get skipped
# (Lakeflow Connect owns them as Streaming Tables) -- but the code itself never
# assumes that in advance. Attempt first, record the real outcome second.
#
# CATALOG below is the real literal GlobalMart's Bronze/Silver run actually uses
# -- harsh_kumar01_npmentorskool_onmicrosoft_com, NOT gbmart (gbmart is a
# separate catalog, used only for Gold -- see Day 7). Your own catalog will be
# named differently; replace this with <your-catalog> in your own notebook.
CATALOG = "harsh_kumar01_npmentorskool_onmicrosoft_com"
SCHEMA  = "bronze"

BRONZE_TABLES = [
    "customers", "orders", "order_items", "products",
    "addresses", "payments", "payment_methods", "returns"
]

for table in BRONZE_TABLES:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    try:
        spark.sql(f"ALTER TABLE {full_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print(f"  CDF enabled : {full_name}")
    except Exception as e:
        print(f"  SKIPPED     : {full_name} — {str(e)[:80]}")

In [ ]:
# Verify — DESCRIBE HISTORY will show a new version where the table property changed.
spark.sql(f"DESCRIBE HISTORY {CATALOG}.bronze.customers") \
    .select("version", "timestamp", "operation") \
    .orderBy("version", ascending=False) \
    .show(5, truncate=False)

Notice `addresses` is **plural** in Bronze — that's the real table name, keep it exactly as-is (Silver's version of this table is spelled differently, see below).

Santosh Kumar ran this exact loop against the real `harsh_kumar01_npmentorskool_onmicrosoft_com` catalog and watched `orders` and `order_items` print `SKIPPED` — exactly what the real pipeline notebook's markdown predicted, just discovered live instead of assumed ahead of time. That's the whole point of the try/except shape: the code finds out, it doesn't guess.

## Enabling CDF on Silver Too

Enable it now, before Silver has any incremental readers, for the same reason Bronze needed it enabled early: turning CDF on **late** means missing whatever changed in the gap.

**Real architecture fact, worth remembering exactly:** Gold (`dim_*` / `fact_sales`) is **not** incremental today — Day 6/7 built it via full overwrite, not incremental `MERGE` — so nothing currently reads a Silver change feed. Enabling CDF on Silver now isn't because today's Gold build depends on it. It's so the option is already open, at zero cost, for later incremental work. Day 10 is where that later work — a real, hands-on SCD1/SCD2 `MERGE` build — actually happens.

In [ ]:
SILVER_TABLES = [
    "customers", "orders", "order_items", "products",
    "address", "payments", "payment_methods", "returns"
]

for table in SILVER_TABLES:
    full_name = f"{CATALOG}.silver.{table}"
    try:
        spark.sql(f"ALTER TABLE {full_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
        print(f"  CDF enabled : {full_name}")
    except Exception as e:
        print(f"  SKIPPED     : {full_name} — {str(e)[:80]}")

Look closely at that list: **`address`, singular**, in Silver — versus **`addresses`, plural**, in Bronze a few cells back. That's not a typo to "fix." It's the real, verbatim table name in GlobalMart's actual Silver schema. S R Sneha is the kind of engineer who'd notice this mismatch immediately and leave it alone rather than "cleaning it up" — renaming a real production table name to look more consistent would break every notebook and job that already references it by its actual name.

## Reading a Change Feed

Once enabled, reading changes since a version is one option pair on a normal batch read — no separate API to learn.

In [ ]:
# Read every change recorded on CATALOG.bronze.products since version 0 (the very
# start). In HOL 2 you'll do this starting from a real "last processed" version
# instead of 0, exactly like a production incremental job would.
cdf_df = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", 0)
        .table(f"{CATALOG}.bronze.products")
)

cdf_df.groupBy("_change_type").count().show()
cdf_df.select("product_id", "discounted_price_inr", "_change_type", "_commit_version").show(10, truncate=False)

## The Same Read, in SQL: `table_changes()`

`readChangeFeed` above is the PySpark option-pair form. Delta also exposes CDF as a plain **SQL table-valued function**, `table_changes()` — same underlying data, same 4 `_change_type` values, useful when you're working in a SQL notebook/dashboard instead of PySpark, or piping a change feed straight into a `CREATE OR REPLACE VIEW`. Both forms read the identical CDF log; pick whichever fits the tool you're already in.

In [ ]:
# table_changes(table_name, start_version [, end_version]) -- same CDF log as the
# readChangeFeed cell above, same version-0-to-latest range, same drop of
# update_preimage. This is the SQL-native way to read a change feed.
spark.sql(f"""
    SELECT product_id, discounted_price_inr, _change_type, _commit_version
    FROM table_changes('{CATALOG}.bronze.products', 0)
    WHERE _change_type != 'update_preimage'
    LIMIT 10
""").display()

## The Real Bronze → Silver MERGE Pattern — A Walkthrough, Not a Live Run

Everything above is real code, safe to run again and again — `ALTER TABLE ... SET TBLPROPERTIES` and read-only `SELECT`s don't create duplicate rows no matter how many times you run them.

What follows is different. This is a **verbatim walkthrough of GlobalMart's actual `10_products_incremental_scd2_merge.py`** — the real Bronze → Silver SCD2 `MERGE` that keeps `harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products` in sync using CDF. We're reading it, line by line, understanding exactly what it does and why — but **not running it** in this ILT.

**Why not run it here?** `ALTER TABLE ... SET TBLPROPERTIES` is idempotent — running it five times leaves the table in exactly the same state as running it once. A `MERGE` + `append` pair like the one below is not. Running it twice would insert a second batch of "new version" rows into a Silver table the whole cohort shares. S R Sneha is the kind of engineer who reads a `MERGE` twice, end to end, before ever running it once — that habit is exactly what this walkthrough is practicing.

**Day 10 is where you build a hands-on, corrected SCD2 `MERGE` yourself**, safely, against your own practice schema — not here. This session's job is to understand the real pattern first, rough edges included.

### Steps 1–2 — Where This Starts: All of Bronze, Then Just the Version Boundary

```python
BRONZE_TABLE = f"{CATALOG}.bronze.products"
SILVER_TABLE = f"{CATALOG}.silver.products"

bronze_df = spark.table(BRONZE_TABLE)
print(f"Total rows currently in Bronze: {bronze_df.count():,}")
```

Step 2 is `DESCRIBE HISTORY` again — the exact command you already ran for real earlier, just pointed at `bronze.products` this time, to find the version number right before the incremental file landed.

```python
LAST_PROCESSED_VERSION = 0   # <-- set this from the history output above
```

**A real, worth-naming-explicitly shortcut in this exact demo script:** `LAST_PROCESSED_VERSION` is a number someone read off the `DESCRIBE HISTORY` output and typed in by hand — not a value pulled back from a stored checkpoint table. That's fine for a one-time demo run with a human watching the screen. It would not be fine unattended: run this notebook twice with the same hardcoded `0`, and CDF would happily hand back "everything since version 1" a second time, re-processing rows that were already merged in. A production version of this pattern would read `LAST_PROCESSED_VERSION` from a control table at the start and write the new value back at the end — the same "advance the marker" step ILT 2's watermark loader did with a timestamp column instead of a Delta version number.

### Step 3 — Take Only the Changes (CDF, Not a Full Re-Scan)

```python
cdf_changes_df = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", LAST_PROCESSED_VERSION + 1)
        .table(BRONZE_TABLE)
        .filter("_change_type != 'update_preimage'")
)
```

Same shape as the CDF read cell you already ran for real, just starting one version **past** `LAST_PROCESSED_VERSION` instead of from the very beginning — and again dropping `update_preimage`, because Step 4 only wants each changed row's current state.

**The real demo scenario this notebook was built around:** 2 price changes (`PRD-00001`, `PRD-00002`) + 1 new product (`PRD-00501`) — 3 changed rows total, regardless of how large `bronze.products` eventually grows. That's the entire point of CDF: the cost of this read tracks *what changed*, not *how big the table is*.

### Step 4 — Process Only These Changed Rows

```python
processed_df = cdf_changes_df.select(
    "product_id", "product_name", "category", "sub_category",
    "actual_price_inr", "discounted_price_inr", "rating", "num_ratings",
    col("specs.power_source").alias("power_source"),
    col("specs.country_of_origin").alias("country_of_origin"),
    col("specs.color_options").alias("color_options"),
    col("specs.is_returnable").alias("is_returnable"),
    col("specs.return_window_days").alias("return_window_days"),
    col("specs.material").alias("material"),
    col("specs.warranty_months").alias("warranty_months"),
    col("specs.weight_kg").alias("weight_kg"),
    col("supplier_info.supplier_id").alias("supplier_id"),
    col("supplier_info.name").alias("supplier_name"),
    col("supplier_info.city").alias("supplier_city"),
    "tags", "last_updated"
)
```

Exactly the same flatten logic Day 5's full-load notebook used on `specs.*` and `supplier_info.*` — just scoped down to 3 changed rows instead of the full 500+-row table.

### Step 5 — Load Into Silver: SCD2, in Two Parts

**The filing-cabinet picture, before any code:** think of `dim_product` like a filing cabinet that keeps every past address a person ever had, not just the current one. When someone moves, the clerk doesn't erase the old card — they stamp it "moved out" with today's date and file a brand-new card marked "current." SCD2 does exactly that to a price change: the old price row gets stamped closed, a new row gets filed as current. Nothing is ever deleted — you can always answer "what was this product's price *last month*," not just "what is it now."

**5a — close out the old version** (update-only, no insert):

```python
silver_table = DeltaTable.forName(spark, SILVER_TABLE)

(silver_table.alias("tgt")
    .merge(
        processed_df.alias("src"),
        "tgt.product_id = src.product_id AND tgt.is_current = true "
        "AND tgt.discounted_price_inr <> src.discounted_price_inr"
    )
    .whenMatchedUpdate(set={
        "is_current": "false",
        "effective_end_date": "current_date()"
    })
    .execute()
)
```

Match key is a plain natural key, `product_id`. There's no `whenNotMatched` clause here at all — this `MERGE` only ever updates existing rows, it never inserts.

**5b — append the new version, unconditionally:**

```python
new_versions_df = processed_df \
    .withColumn("effective_start_date", current_date()) \
    .withColumn("effective_end_date", lit(None).cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("product_sk",
        sha2(concat_ws("|", col("product_id"), col("effective_start_date").cast("string")), 256)
    ) \
    .withColumn("discount_pct",
        round((col("actual_price_inr") - col("discounted_price_inr")) / col("actual_price_inr") * 100, 2)
    ) \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select(
        "product_sk", "product_id", "product_name", "category", "sub_category",
        "actual_price_inr", "discounted_price_inr", "discount_pct",
        "rating", "num_ratings",
        "power_source", "country_of_origin", "color_options", "is_returnable",
        "return_window_days", "material", "warranty_months", "weight_kg",
        "supplier_id", "supplier_name", "supplier_city", "tags",
        "is_current", "effective_start_date", "effective_end_date",
        "last_updated", "_silver_updated_at"
    )

new_versions_df.write.format("delta").mode("append").saveAsTable(SILVER_TABLE)
```

The surrogate key formula — `sha2(concat_ws("|", product_id, effective_start_date), 256)` — is the exact same `dim_product` SCD2 key formula already taught in Day 6/7. Every row in `processed_df` gets appended as a fresh, `is_current = true` row. This single append covers **both** cases at once: a changed product (a new price version) and a genuinely new product (5a never matched it, so this append is the only row it gets).

### A Real Limitation in This Exact Code — Worth Discussing, Not Fixing Here

Look closely at what each half of Step 5 actually checks:

- **5a** only closes out the old row when `tgt.discounted_price_inr <> src.discounted_price_inr` — the price, specifically.
- **5b** appends **every** row in `processed_df`, unconditionally — price change or not.

Those two conditions don't match, and that's a real, latent bug in this exact demo script. If a CDF change touched `category`, `rating`, `tags`, or anything **other** than `discounted_price_inr`, 5a finds nothing to close out (the price didn't change, so its merge condition never matches) — but 5b still appends that row as a brand-new `is_current = true` version anyway. The result: **two** `is_current = true` rows for the same `product_id`, and neither one was ever closed. A double-current row.

Saket Ranjan is the one who'd catch this in review: tracing why a `dim_product` row seems to have two `is_current = true` versions after a category update that never touched price, and working backward to the mismatched conditions above.

This ILT is showing you the real code GlobalMart's pipeline actually runs — rough edges included — not a polished textbook example. **We are not fixing it here, and we're not quietly pretending it isn't there.** Day 10 is where you build a corrected, hands-on SCD2 `MERGE` yourself — one where the close-out condition and the insert condition are kept honestly in sync with each other. For today, the goal is just to see the gap clearly and understand exactly why it exists.

### Step 6 — Verify

```python
df = spark.table(SILVER_TABLE)
df.filter(col("product_id").isin(["PRD-00001", "PRD-00002", "PRD-00501"])) \
  .select("product_sk", "product_id", "discounted_price_inr", "is_current",
          "effective_start_date", "effective_end_date") \
  .orderBy("product_id", "effective_start_date") \
  .display()
```

In the clean case, each changed `product_id` (`PRD-00001`, `PRD-00002`) shows **2 rows**: an old version (`is_current = false`, `effective_end_date` set) and a new version (`is_current = true`, `effective_end_date = NULL`). The new product (`PRD-00501`) shows a single `is_current = true` row, since there was nothing to close out. The limitation above is exactly the scenario where this clean 2-row picture breaks down.

## The Same Pattern, a Second Time: `dim_customer`

GlobalMart's real `11_customers_incremental_scd2_merge.py` applies the **identical** six-step shape to `dim_customer` — worth seeing once, briefly, to prove this is a generalizable pattern, not a one-off quirk of `products`.

- **Step 3, CDF read:** same shape exactly — `readChangeFeed`, `startingVersion`, drop `update_preimage`.
- **Step 4, real cleanup:** filters `CustomerID IS NOT NULL AND Email IS NOT NULL` before doing anything else. That's not defensive boilerplate — it's a real cross-file interaction: a `customer_consent_*.csv` file lands in the same Bronze source folder, and its rows would otherwise show up in this same CDF read even though they're a completely different entity. Remalli Kamal Kumar is the one who'd trace an unexpected null-heavy row back to that consent file rather than assuming it's bad customer data. The same cell also computes `age` and `customer_tenure_days`, and renames PascalCase source columns (`CustomerID`, `Email`, ...) to snake_case (`customer_id`, `email`, ...).
- **Step 5a, close-out condition:** `tgt.email <> src.email` — same shape as `products`' price check, same limitation. An attribute change that isn't `email` (say, `phone_number`) would append a new row without closing the old one, for exactly the same reason as the products case above.
- **Step 5b, append:** identical pattern — `sha2(concat_ws("|", customer_id, effective_start_date), 256)` as the surrogate key, unconditional append of every CDF row.

Same strengths, same rough edge, applied to a second dimension. That's the real takeaway: this MERGE shape is GlobalMart's actual reusable pattern for CDF-driven SCD2 — and the double-current-row limitation isn't a `products`-specific mistake, it's a property of the *pattern itself*, wherever it gets copy-pasted without the close-out and insert conditions being kept in sync.

## Key Takeaways

- CDF is a Delta table property, enabled once with `ALTER TABLE ... SET TBLPROPERTIES (delta.enableChangeDataFeed = true)`. Every `INSERT`/`UPDATE`/`DELETE` afterward is recorded automatically — including deletes, which is exactly the gap ILT 2's watermark strategy couldn't close.
- `_change_type` has 4 values; always drop `update_preimage` before merging — you want each row's current state, not its "before" snapshot.
- Two ways to read the same CDF log: `readChangeFeed` (PySpark option pair) and `table_changes()` (SQL table-valued function) — same data, pick whichever fits the tool you're in.
- Bronze's real table list is 8 tables (`addresses`, plural); Silver's real list is also 8 tables (`address`, singular) — a real, intentional asymmetry in GlobalMart's actual schema, not a typo.
- Gold is **not** incremental today — full overwrite, per Day 6/7 — so Silver's CDF is enabled ahead of need, not because anything reads it yet.
- The real `dim_product`/`dim_customer` SCD2 `MERGE` pattern is: 5a closes out the old `is_current` row on a matched attribute change (update-only `MERGE`, no insert clause), 5b unconditionally appends every CDF row as a new version. That mismatch — 5a is conditional, 5b isn't — is a real, named limitation: a double-current row is possible when an *unwatched* attribute changes.
- This ILT is a **read-only walkthrough**. Nothing here writes to `harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products` or `.silver.customers`. Day 10 is where you build and fix this pattern yourself, hands-on.

## Self-Check

- [ ] I can explain, in one sentence, why CDF closes the delete blind spot that watermark loading couldn't.
- [ ] I can name the 4 `_change_type` values and say which one always gets filtered out before a merge, and why.
- [ ] I can say, without looking, which Bronze/Silver table name is plural (`addresses`) and which is singular (`address`).
- [ ] I can explain why Gold's CDF-on-Silver setup isn't "wasted" even though nothing reads it today.
- [ ] I can describe the filing-cabinet analogy for SCD2 in my own words — close old row, append new row, never delete.
- [ ] I can point to the exact line in Step 5a and Step 5b where the double-current-row limitation comes from, and explain it to someone else.
- [ ] I know this ILT was read-only, and I know Day 10 is where the hands-on, corrected SCD2 build happens.